In [ ]:
# CELL 1 — Install Dependencies
!nvidia-smi
!pip install -q openai-whisper
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q torch torchvision torchaudio
!pip install -q transformers timm pillow opencv-python sentencepiece
!pip install -q fastapi uvicorn nest-asyncio pyngrok python-multipart requests
!apt-get update -qq && apt-get install -y -qq ffmpeg
print('All dependencies installed')


In [ ]:
# CELL 2 — Imports & Folder Setup
import os, json, subprocess, shutil, uuid, threading, requests
import numpy as np
import cv2
from PIL import Image
import torch

import whisper
import faiss
from sentence_transformers import SentenceTransformer
from transformers import BlipProcessor, BlipForConditionalGeneration
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import nest_asyncio
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Request
from fastapi.responses import JSONResponse, FileResponse, Response
from fastapi.middleware.cors import CORSMiddleware
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.requests import Request as StarletteRequest
from pyngrok import ngrok

# Folder layout — keep original names for Colab cells; BASE alias used by API
os.makedirs("video", exist_ok=True)
os.makedirs("frames", exist_ok=True)
os.makedirs("transcripts", exist_ok=True)
os.makedirs("embeddings", exist_ok=True)
os.makedirs("clips", exist_ok=True)

BASE = "/content"   # so API paths like BASE/video/input.mp4 = "video/input.mp4"

print("Folders created")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Imports done. Device:', DEVICE)


In [ ]:
# CELL 3 — Load All Models
# Whisper + BLIP + SentenceTransformer + Flan-T5 (raw tokenizer, NOT hf pipeline)
# First run takes 3-5 minutes while models download.

print("Loading Whisper base...")
model = whisper.load_model("base")          # named 'model' to match original Colab cells
print("  Whisper loaded")

print("Loading BLIP captioning...")
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(DEVICE)
print("  BLIP loaded")

print("Loading SentenceTransformer...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("  SentenceTransformer loaded")

# Flan-T5 — loaded as raw model+tokenizer, NOT via hf pipeline.
# The transformers text2text-generation pipeline ignores max_new_tokens
# and can crash on certain Colab CUDA builds. Direct .generate() is reliable.
print("Loading Flan-T5-base...")
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
flan_model     = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(DEVICE)
flan_model.eval()

def flan_generate(prompt: str, max_new_tokens: int = 120) -> str:
    """Run Flan-T5 inference directly. Truncates input to 512 tokens (model hard limit)."""
    inputs = flan_tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(DEVICE)
    with torch.no_grad():
        output_ids = flan_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
    return flan_tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

print("  Flan-T5 loaded")
print("\nAll models ready!")


In [ ]:
# CELL 4 — LLM Provider Config & Helper Functions
# ── Choose your LLM ──────────────────────────────────────────────────────────
#   'groq'      — FREE  — https://console.groq.com
#   'local'     — FREE, offline, weaker quality (Flan-T5)
API_PROVIDER = "groq"   # <── change this

GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERE"    # <── paste from console.groq.com
GROQ_MODEL   = "llama-3.1-8b-instant"
GROQ_URL     = "https://api.groq.com/openai/v1/chat/completions"

L2_THRESHOLD = 1.2

CONTEXT_CHAR_LIMITS = {
    "groq":      1000,
    "local":      500,
}

SUMMARY_CHUNK_CHARS = {
    "groq":      2000,
    "local":      600,
}

QA_SYSTEM_PROMPT = """You are an AI assistant for a video question-answering system.
Answer ONLY using the retrieved context provided. Keep your answer to 1-2 clear sentences.
If the context does not contain the answer, reply with exactly:
Not available in the video content.
Never mention 'context', 'transcript', or 'based on'. Just answer directly."""

SUMMARY_SYSTEM_PROMPT = """You are a video summarisation assistant.
Given transcript text from a video, write a clear, well-structured summary in flowing prose paragraphs.
Cover: main topic, key concepts, important examples or demonstrations, notable facts or names, and conclusions.
Do not use bullet points. Do not say 'the transcript says'. Just summarise directly."""

NOT_FOUND_MSG = (
    "This question does not appear to be covered in the video content. "
    "Please ask something related to what is discussed in the video."
)


def call_llm_qa(query: str, context: str) -> str:
    limit    = CONTEXT_CHAR_LIMITS.get(API_PROVIDER, 800)
    ctx      = context[:limit]
    user_msg = f"Retrieved context:\n{ctx}\n\nQuestion: {query}\nAnswer:"

    if API_PROVIDER == "groq":
        headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
        payload = {
            "model": GROQ_MODEL,
            "messages": [
                {"role": "system", "content": QA_SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            "max_tokens": 150, "temperature": 0.1,
        }
        r = requests.post(GROQ_URL, headers=headers, json=payload, timeout=15)
        if not r.ok:
            print(f"Groq QA error {r.status_code}: {r.text}")
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip()

    else:  # local Flan-T5
        prompt = (
            "Answer in one sentence using only the context below. "
            "If not in context, say: Not available in the video content.\n\n"
            f"Context: {ctx}\nQuestion: {query}\nAnswer:"
        )
        return flan_generate(prompt, max_new_tokens=80)


def call_llm_summary(text_chunk: str) -> str:
    if API_PROVIDER == "groq":
        headers = {"Authorization": f"Bearer {GROQ_API_KEY}", "Content-Type": "application/json"}
        payload = {
            "model": GROQ_MODEL,
            "messages": [
                {"role": "system", "content": SUMMARY_SYSTEM_PROMPT},
                {"role": "user",   "content": f"Transcript:\n{text_chunk}\n\nSummarise this section:"},
            ],
            "max_tokens": 400, "temperature": 0.3,
        }
        r = requests.post(GROQ_URL, headers=headers, json=payload, timeout=20)
        r.raise_for_status()
        return r.json()["choices"][0]["message"]["content"].strip()

    else:  # local Flan-T5
        prompt = f"Summarise this video section in 3-5 sentences:\n{text_chunk[:500]}"
        return flan_generate(prompt, max_new_tokens=150)


def generate_answer(query: str, context: str) -> str:
    try:
        answer = call_llm_qa(query, context)
        for phrase in ["based on the context", "according to the transcript",
                       "the context states", "the context mentions",
                       "based on the retrieved", "from the context"]:
            answer = answer.replace(phrase, "").strip(", .")
        return answer
    except Exception as e:
        print(f"[QA] API error ({API_PROVIDER}): {e} — falling back to Flan-T5")
        ctx = context[:CONTEXT_CHAR_LIMITS.get("local", 500)]
        prompt = (
            "Answer in one sentence using only the context. "
            "If not in context, say: Not available in the video content.\n\n"
            f"Context: {ctx}\nQuestion: {query}\nAnswer:"
        )
        return flan_generate(prompt, max_new_tokens=80)


print(f"LLM config ready. Provider: {API_PROVIDER}")


---
## 🎬 Part A — Colab Mode
*Run cells 5 → 13 to upload a video, process it, and interact with it directly in this notebook.*


In [ ]:
# CELL 5 — Upload Video (Colab)
from google.colab import files

uploaded = files.upload()

for name in uploaded:
    os.rename(name, "video/input.mp4")

print("Video saved as video/input.mp4")


In [ ]:
# CELL 6 — Whisper Transcription
result = model.transcribe(
    "video/input.mp4",
    word_timestamps=True
)

segments = []
for seg in result["segments"]:
    segments.append({
        "text": seg["text"].strip(),
        "start": seg["start"],
        "end": seg["end"]
    })

with open("transcripts/transcript.json", "w") as f:
    json.dump(segments, f, indent=2)

print("Transcription completed")
print("Total segments:", len(segments))


In [ ]:
# CELL 7 — Text Embeddings & FAISS Index
texts = [seg["text"] for seg in segments]
text_embeddings = embed_model.encode(texts, convert_to_numpy=True)

dim = text_embeddings.shape[1]
text_index = faiss.IndexFlatL2(dim)
text_index.add(text_embeddings)

faiss.write_index(text_index, "embeddings/text.index")

print("Text FAISS index created")


In [ ]:
# CELL 8 — Text Embeddings Proof
print("\n================ TEXT EMBEDDINGS PROOF ================")

print("Total text embeddings in FAISS:", text_index.ntotal)
print("Embedding dimension:", text_index.d)

print("\n--- Sample Text Embeddings (First 3) ---")
for i in range(5):
    print(f"\nSegment {i+1}")
    print("Text:", segments[i]["text"])
    print("Timestamp:", segments[i]["start"], "-", segments[i]["end"])
    print("Embedding (first 10 values):", text_embeddings[i][:10])


In [ ]:
# CELL 9 — Extract Frames
cap = cv2.VideoCapture("video/input.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)

frame_interval = int(fps * 2)  # 1 frame every 2 seconds
frame_id = 0
saved = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    if frame_id % frame_interval == 0:
        timestamp = frame_id / fps
        cv2.imwrite(f"frames/frame_{timestamp:.2f}.jpg", frame)
        saved += 1

    frame_id += 1

cap.release()
print(f"Extracted {saved} frames")


In [ ]:
# CELL 10 — BLIP Visual Captions
visual_segments = []

for img in sorted(os.listdir("frames")):
    image = Image.open(f"frames/{img}").convert("RGB")
    inputs = processor(image, return_tensors="pt").to(DEVICE)
    out = blip_model.generate(**inputs, max_new_tokens=30)
    caption = processor.decode(out[0], skip_special_tokens=True)

    timestamp = float(img.split("_")[1].replace(".jpg", ""))
    visual_segments.append({
        "text": caption,
        "start": timestamp,
        "end": timestamp + 1
    })

with open("transcripts/visual_segments.json", "w") as f:
    json.dump(visual_segments, f, indent=2)

print("Visual captions generated:", len(visual_segments))


In [ ]:
# CELL 11 — Visual Embeddings & FAISS Index
visual_texts = [v["text"] for v in visual_segments]
visual_embeddings = embed_model.encode(visual_texts, convert_to_numpy=True)

visual_index = faiss.IndexFlatL2(dim)
visual_index.add(visual_embeddings)

faiss.write_index(visual_index, "embeddings/visual.index")

print("Visual FAISS index created")


In [ ]:
# CELL 12 — Visual Embeddings Proof
print("\n================ VISUAL EMBEDDINGS PROOF ================")

print("Total visual embeddings in FAISS:", visual_index.ntotal)
print("Embedding dimension:", visual_index.d)

print("\n--- Sample Visual Embeddings (First 3) ---")
for i in range(5):
    print(f"\nVisual Segment {i+1}")
    print("Caption:", visual_segments[i]["text"])
    print("Timestamp:", visual_segments[i]["start"], "-", visual_segments[i]["end"])
    print("Embedding (first 10 values):", visual_embeddings[i][:10])


In [ ]:
# CELL 13 — Combined Vector Database (Audio + Visual)
print("\n================ COMBINED VECTOR DATABASE (AUDIO + VISUAL) ================")

combined_db = []

# Add AUDIO / TEXT embeddings
for i in range(len(segments)):
    combined_db.append({
        "modality": "Audio",
        "text": segments[i]["text"],
        "start": segments[i]["start"],
        "end": segments[i]["end"],
        "embedding": text_embeddings[i]
    })

# Add VISUAL embeddings
for i in range(len(visual_segments)):
    combined_db.append({
        "modality": "Visual",
        "text": visual_segments[i]["text"],
        "start": visual_segments[i]["start"],
        "end": visual_segments[i]["end"],
        "embedding": visual_embeddings[i]
    })

print(f"Total entries in combined vector DB: {len(combined_db)}")

print("\n--- SAMPLE COMBINED ENTRIES (First 5) ---")
for i in range(min(10, len(combined_db))):
    item = combined_db[i]
    print(f"\nEntry {i+1}")
    print("Modality:", item["modality"])
    print("Text/Caption:", item["text"])
    print("Timestamp:", item["start"], "-", item["end"])
    print("Embedding (first 10 values):", item["embedding"][:10])


In [ ]:
# CELL 14 — Full Video Speech Summary (Chunked)
# ===============================
# FULL SPEECH SUMMARY (CHUNKED)
# ===============================

def chunk_text(text, chunk_size=500):
    words = text.split()
    for i in range(0, len(words), chunk_size):
        yield " ".join(words[i:i + chunk_size])

# 1. Combine full speech
full_speech = " ".join([seg["text"] for seg in segments])

print("Total words in video speech:", len(full_speech.split()))

# 2. Create chunks
speech_chunks = list(chunk_text(full_speech, chunk_size=400))
print("Total chunks:", len(speech_chunks))

chunk_summaries = []

# 3. Summarize each chunk using the configured LLM (Groq / Anthropic / Flan-T5)
for i, chunk in enumerate(speech_chunks):
    summary = call_llm_summary(chunk)
    chunk_summaries.append(summary)
    print(f"Chunk {i+1} summarized")

# 4. Combine all summaries
full_summary = "\n\n".join(chunk_summaries)

print("\n================ FULL VIDEO SPEECH SUMMARY ================\n")
print(full_summary)


In [ ]:
# CELL 15 — Q&A
query = input("Ask your question: ")

query_emb = embed_model.encode([query], convert_to_numpy=True)

# Text retrieval
D_t, I_t = text_index.search(query_emb, k=2)
text_context = " ".join(segments[i]["text"] for i in I_t[0])

# Visual retrieval
D_v, I_v = visual_index.search(query_emb, k=2)
visual_context = " ".join(visual_segments[i]["text"] for i in I_v[0])

combined_context = text_context + " " + visual_context

answer = generate_answer(query, combined_context)

print("\n✅ FINAL ANSWER:")
print(answer)


In [ ]:
# CELL 16 — Extract & Display Answer Clip
import subprocess
from IPython.display import Video, display

best_segment = segments[I_t[0][0]]

start = max(0, best_segment["start"] - 0.9)
end = best_segment["end"] + 0.5

subprocess.run([
    "ffmpeg", "-y",
    "-i", "video/input.mp4",
    "-ss", str(start),
    "-to", str(end),
    "clips/answer_clip.mp4"
])

display(Video("clips/answer_clip.mp4", embed=True))


---
## 🌐 Part B — Web Backend (FastAPI + ngrok)
*Run cells 17 → 19 to expose the same pipeline as a REST API for your website.*
*You can run Part B independently (skip Part A) — the API processes videos uploaded via POST /upload.*


In [ ]:
# CELL 17 — Pipeline State & Helper Functions (used by the API)

state = {
    "ready": False, "processing": False, "error": None,
    "video_path": None, "video_name": "", "duration": 0,
    "segments": [], "visual_segments": [],
    "text_index": None, "visual_index": None,
    "stage": "",
}


def get_video_duration(path: str) -> float:
    cap = cv2.VideoCapture(path)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 25
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()
    return round(frames / fps, 1)


def api_transcribe_audio(path: str):
    result = model.transcribe(path, word_timestamps=True)
    segs   = []
    for seg in result["segments"]:
        text = seg["text"].strip()
        if text:
            segs.append({"text": text, "start": seg["start"], "end": seg["end"]})
    return segs


def api_extract_frames_and_caption(path: str, interval_secs: int = 2):
    frames_dir = "frames"
    shutil.rmtree(frames_dir, ignore_errors=True)
    os.makedirs(frames_dir, exist_ok=True)

    cap = cv2.VideoCapture(path)
    fps  = cap.get(cv2.CAP_PROP_FPS) or 25
    step = max(1, int(fps * interval_secs))
    frame_id, frame_data = 0, []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        if frame_id % step == 0:
            ts    = frame_id / fps
            fpath = f"{frames_dir}/frame_{ts:.2f}.jpg"
            cv2.imwrite(fpath, frame)
            frame_data.append((ts, fpath))
        frame_id += 1
    cap.release()

    visual_segs = []
    for ts, fpath in frame_data:
        try:
            img    = Image.open(fpath).convert("RGB")
            inputs = processor(img, return_tensors="pt").to(DEVICE)
            out    = blip_model.generate(**inputs, max_new_tokens=40, num_beams=3)
            caption = processor.decode(out[0], skip_special_tokens=True)
            visual_segs.append({"text": caption, "start": ts, "end": ts + interval_secs})
        except Exception as e:
            print(f"  BLIP skip {ts:.1f}s: {e}")
    return visual_segs


def api_build_faiss_index(segs):
    if not segs:
        return None
    texts = [s["text"] for s in segs]
    embs  = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    index = faiss.IndexFlatL2(embs.shape[1])
    index.add(embs)
    return index


def api_retrieve_context(query: str, k: int = 3):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    hits  = []

    if state["text_index"] is not None:
        D, I = state["text_index"].search(q_emb, k=min(k, state["text_index"].ntotal))
        for d, i in zip(D[0], I[0]):
            hits.append((float(d), state["segments"][i], "audio"))

    if state["visual_index"] is not None:
        D, I = state["visual_index"].search(q_emb, k=min(k, state["visual_index"].ntotal))
        for d, i in zip(D[0], I[0]):
            hits.append((float(d), state["visual_segments"][i], "visual"))

    if not hits:
        return None, None, 999.0

    hits.sort(key=lambda x: x[0])
    best_dist  = hits[0][0]
    best_audio = next((seg for d, seg, m in hits if m == "audio"), hits[0][1])

    seen, parts = set(), []
    for _, seg, _ in hits[: k * 2]:
        t = seg["text"].strip()
        if t not in seen:
            parts.append(t)
            seen.add(t)

    return " ".join(parts), best_audio, best_dist


def api_extract_clip(segment, clip_path: str, pad_start: float = 0.8, pad_end: float = 1.5):
    start = max(0.0, segment["start"] - pad_start)
    end   = segment["end"] + pad_end
    cmd   = [
        "ffmpeg", "-y", "-i", state["video_path"],
        "-ss", str(start), "-to", str(end),
        "-c:v", "libx264", "-c:a", "aac",
        "-movflags", "+faststart", clip_path,
    ]
    r = subprocess.run(cmd, capture_output=True)
    return clip_path if (r.returncode == 0 and os.path.exists(clip_path)) else None


def process_video_bg(video_path: str, video_name: str):
    try:
        state.update({
            "processing": True, "ready": False, "error": None,
            "video_path": video_path, "video_name": video_name,
            "segments": [], "visual_segments": [], "stage": "upload",
        })
        state["duration"] = get_video_duration(video_path)
        print(f"[Pipeline] {video_name} | {state['duration']}s")

        state["stage"] = "whisper"
        print("[1/4] Whisper transcription...")
        state["segments"] = api_transcribe_audio(video_path)
        print(f"      {len(state['segments'])} segments")

        state["stage"] = "blip"
        print("[2/4] BLIP captioning...")
        state["visual_segments"] = api_extract_frames_and_caption(video_path)
        print(f"      {len(state['visual_segments'])} visual segments")

        state["stage"] = "faiss"
        print("[3/4] Building FAISS indices...")
        state["text_index"]   = api_build_faiss_index(state["segments"])
        state["visual_index"] = api_build_faiss_index(state["visual_segments"])

        state.update({"ready": True, "processing": False, "stage": "ready"})
        print("[4/4] Ready!")
    except Exception as e:
        import traceback
        state.update({"error": str(e), "processing": False, "stage": "error"})
        traceback.print_exc()


print(f"Pipeline helpers ready. API provider: {API_PROVIDER}")


In [ ]:
# CELL 18 — FastAPI App & Endpoints

app = FastAPI(title="SummariV")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

class NgrokBypassMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request: StarletteRequest, call_next):
        response = await call_next(request)
        response.headers["ngrok-skip-browser-warning"] = "1"
        response.headers["Access-Control-Allow-Origin"] = "*"
        response.headers["Access-Control-Allow-Headers"] = "*"
        response.headers["Access-Control-Allow-Methods"] = "*"
        return response

app.add_middleware(NgrokBypassMiddleware)

NOT_FOUND_MSG = (
    "This question does not appear to be covered in the video content. "
    "Please ask something related to what is discussed in the video."
)


@app.options("/{full_path:path}")
async def preflight_handler(full_path: str, request: Request):
    return Response(
        status_code=200,
        headers={
            "Access-Control-Allow-Origin": "*",
            "Access-Control-Allow-Methods": "GET, POST, PUT, DELETE, OPTIONS",
            "Access-Control-Allow-Headers": "*",
            "ngrok-skip-browser-warning": "1",
        },
    )


@app.get("/health")
def health():
    return {"status": "ok", "api_provider": API_PROVIDER}


@app.post("/upload")
async def upload(video: UploadFile = File(...)):
    if state["processing"]:
        raise HTTPException(409, "Already processing a video. Please wait.")
    video_path = "video/input.mp4"
    with open(video_path, "wb") as f:
        f.write(await video.read())
    shutil.rmtree("clips", ignore_errors=True)
    os.makedirs("clips", exist_ok=True)
    threading.Thread(
        target=process_video_bg,
        args=(video_path, video.filename),
        daemon=True,
    ).start()
    return {"message": "Processing started", "filename": video.filename}


@app.get("/status")
def status():
    return {
        "ready":           state["ready"],
        "processing":      state["processing"],
        "error":           state["error"],
        "stage":           state["stage"],
        "video_name":      state["video_name"],
        "duration":        state["duration"],
        "audio_segments":  len(state["segments"]),
        "visual_segments": len(state["visual_segments"]),
    }


@app.post("/query")
async def query(q: str = Form(...)):
    if not state["ready"]:
        msg = "Still processing video..." if state["processing"] else "No video loaded."
        raise HTTPException(400, msg)
    if not q.strip():
        raise HTTPException(422, "Query is empty.")

    context, best_seg, best_dist = api_retrieve_context(q.strip(), k=3)

    if context is None or best_dist > L2_THRESHOLD:
        return JSONResponse({
            "answer": NOT_FOUND_MSG, "found": False,
            "confidence": round(max(0.0, 1.0 - best_dist / 2.0), 3),
            "clip_id": None, "timestamp_start": None, "timestamp_end": None,
            "context_preview": "",
        })

    answer = generate_answer(q.strip(), context)

    cant_answer = ["not available in the video", "not in the video", "not found",
                   "not mentioned", "cannot answer", "no information"]
    if any(p in answer.lower() for p in cant_answer):
        return JSONResponse({
            "answer": NOT_FOUND_MSG, "found": False,
            "confidence": round(max(0.0, 1.0 - best_dist / 2.0), 3),
            "clip_id": None, "timestamp_start": None, "timestamp_end": None,
            "context_preview": context[:300],
        })

    clip_id   = str(uuid.uuid4())[:8]
    clip_path = f"clips/{clip_id}.mp4"
    clip_ok   = api_extract_clip(best_seg, clip_path)

    return JSONResponse({
        "answer": answer, "found": True,
        "confidence": round(max(0.0, 1.0 - best_dist / 2.0), 3),
        "clip_id":         clip_id if clip_ok else None,
        "timestamp_start": round(best_seg["start"], 2),
        "timestamp_end":   round(best_seg["end"], 2),
        "context_preview": context[:300],
    })


@app.get("/clip/{clip_id}")
def get_clip(clip_id: str):
    if not clip_id.replace("-", "").replace("_", "").isalnum():
        raise HTTPException(400, "Invalid clip id")
    path = f"clips/{clip_id}.mp4"
    if not os.path.exists(path):
        raise HTTPException(404, "Clip not found")
    return FileResponse(path, media_type="video/mp4", headers={"Accept-Ranges": "bytes"})


@app.get("/summary")
def get_summary():
    if not state["ready"]:
        raise HTTPException(400, "No video processed yet.")

    segs = state["segments"]
    if not segs:
        raise HTTPException(400, "No transcript segments available.")

    full_text  = " ".join(
        s["text"].strip()
        for s in sorted(segs, key=lambda x: x["start"])
    )
    chunk_size = SUMMARY_CHUNK_CHARS.get(API_PROVIDER, 2000)

    if len(full_text) <= chunk_size:
        try:
            return JSONResponse({"summary": call_llm_summary(full_text)})
        except Exception as e:
            print(f"[Summary] single-shot failed: {e}")
            raise HTTPException(500, str(e))

    words   = full_text.split()
    chunks, current, cur_len = [], [], 0
    for word in words:
        current.append(word)
        cur_len += len(word) + 1
        if cur_len >= chunk_size and word.endswith("."):
            chunks.append(" ".join(current))
            current, cur_len = [], 0
    if current:
        chunks.append(" ".join(current))

    print(f"[Summary] {len(full_text)} chars → {len(chunks)} chunks")

    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        try:
            print(f"  Chunk {i+1}/{len(chunks)}...")
            chunk_summaries.append(call_llm_summary(chunk))
        except Exception as e:
            print(f"  Chunk {i+1} failed: {e}")

    combined = "\n\n".join(s for s in chunk_summaries if s)

    if len(chunks) > 1 and combined:
        try:
            combine_prompt = (
                "Below are summaries of consecutive sections of a video. "
                "Combine them into a single cohesive summary of the whole video:\n\n"
                + combined
            )
            final = call_llm_summary(combine_prompt[:chunk_size])
        except Exception as e:
            print(f"[Summary] combine pass failed: {e} — using joined summaries")
            final = combined
    else:
        final = combined

    return JSONResponse({"summary": final or "No summary could be generated."})


print("FastAPI app ready.")
print("Endpoints: /health  /upload  /status  /query  /clip/{id}  /summary")


In [ ]:
# CELL 19 — Start ngrok Tunnel + uvicorn Server
# After this cell prints the ngrok URL, paste it into your frontend.
import nest_asyncio, asyncio, time, re, urllib.request
nest_asyncio.apply()

NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # <── from https://dashboard.ngrok.com/get-started/your-authtoken

# Fix ngrok YAML config (Colab sometimes writes a broken one)
config_path = "/root/.config/ngrok/ngrok.yml"
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    f.write(f'version: "2"\nauthtoken: {NGROK_AUTH_TOKEN}\n')
print("ngrok config written.")

# Kill any stale processes
subprocess.run(["pkill", "-f", "ngrok"], capture_output=True)
subprocess.run(["fuser", "-k", "4040/tcp"], capture_output=True)
time.sleep(2)

# Start ngrok
ngrok_proc = subprocess.Popen(
    ["ngrok", "http", "8000", "--log", "stdout", "--log-format", "json"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

# Wait for tunnel URL
NGROK_URL = None
print("Waiting for ngrok tunnel...")
for _ in range(40):
    line = ngrok_proc.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    try:
        data = json.loads(line)
        if data.get("url"):
            NGROK_URL = data["url"]
            break
        m = re.search(r"url=(https://[^\s\"]+)", line)
        if m:
            NGROK_URL = m.group(1)
            break
    except Exception:
        pass

# Fallback: query local ngrok API
if not NGROK_URL:
    time.sleep(3)
    try:
        with urllib.request.urlopen("http://localhost:4040/api/tunnels") as r:
            tunnels = json.loads(r.read())["tunnels"]
            for t in tunnels:
                if t["public_url"].startswith("https"):
                    NGROK_URL = t["public_url"]
                    break
    except Exception as e:
        print("Fallback failed:", e)

print("=" * 55)
print(f"  ngrok URL  : {NGROK_URL}")
print(f"  Health     : {NGROK_URL}/health")
print(f"  Provider   : {API_PROVIDER}")
print("=" * 55)

# Start uvicorn (this cell blocks — interrupt to stop the server)
config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
server = uvicorn.Server(config)
server.install_signal_handlers = lambda: None
asyncio.get_event_loop().run_until_complete(server.serve())
